# CPMM TPU Code LLM

This notebook trains a pure post-transformer CPMM-style Python code model on Colab TPU with Google Drive checkpointing and exact resume support.

Notebook flow:
- install TPU/JAX dependencies
- mount Google Drive
- prepare tokenizer and JSONL training shards
- initialize or restore the CPMM JAX model
- pretrain on Python code with next-token and graph-memory objectives
- optionally instruction-tune for GPT-style chat/code behavior
- run long-context code evaluations and sample generations

In [7]:
!cd /content/zap && git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 922 bytes | 922.00 KiB/s, done.
From https://github.com/uditgoyal9911/cpmm
   7d9f378..30040c4  main       -> origin/main
Updating 7d9f378..30040c4
Fast-forward
 colab/train_cpm_code_llm.ipynb | 38 +++++++++++++++++++++++++++++++-------
 1 file changed, 31 insertions(+), 7 deletions(-)


In [2]:
# Cell 1: Clone repo + install dependencies
# NOTE: Switch runtime to TPU before running (Runtime > Change runtime type > TPU)

import subprocess, sys

# Clone repo if not already present
result = subprocess.run(['test', '-d', '/content/zap'], capture_output=True)
if result.returncode != 0:
    subprocess.run(['git', 'clone', 'https://github.com/uditgoyal9911/cpmm.git', '/content/zap'], check=True)

# Install JAX TPU and all dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'jax[tpu]', '-f', 'https://storage.googleapis.com/jax-releases/libtpu_releases.html'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'flax', 'optax', 'orbax-checkpoint', 'datasets', 'sentencepiece', 'etils'], check=True)

import os, sys
from pathlib import Path

REPO_ROOT = Path('/content/zap')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import jax
print('JAX version:', jax.__version__)
print('Devices:', jax.devices())
print('Device type:', jax.devices()[0].device_kind)

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


JAX version: 0.9.2
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
Device type: TPU v5 lite


In [8]:
# Cell 2: Mount Google Drive and set up experiment config

import sys
if '/content/zap' not in sys.path:
    sys.path.insert(0, '/content/zap')

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from cpmm_jax.config import ExperimentConfig, save_config

config = ExperimentConfig()

# All persistent data lives in Drive so it survives Colab disconnects
DRIVE_ROOT = '/content/drive/MyDrive/cpmm_code_llm'
config.checkpoint.drive_root = DRIVE_ROOT
config.checkpoint.checkpoint_dir = f'{DRIVE_ROOT}/checkpoints'
config.checkpoint.metadata_path = f'{DRIVE_ROOT}/checkpoints/metadata.json'
config.data.cache_dir = f'{DRIVE_ROOT}/cache'
config.data.tokenizer_path = f'{DRIVE_ROOT}/tokenizer.model'
config.data.train_shards_glob = f'{DRIVE_ROOT}/data/train/*.jsonl'
config.data.eval_shards_glob = f'{DRIVE_ROOT}/data/eval/*.jsonl'

# Use a more accessible dataset that doesn't require HF login
config.data.dataset_name = 'codeparrot/github-code'
config.data.language = 'Python'

# Colab-safe model size: fits in TPU HBM with batch_size=64
config.model.d_model = 256
config.model.num_slots = 6
config.model.num_graph_symbols = 64
config.model.num_graph_values = 64
config.data.parser_max_nodes = 64  # 64x64 graph = 16KB vs 256x256 = 256KB per record
config.model.max_seq_len = 512
config.train.global_batch_size = 32
config.train.total_steps = 5000
config.train.warmup_steps = 100
config.train.log_every = 10
config.train.eval_every = 100
config.train.save_every = 100

Path(config.checkpoint.drive_root).mkdir(parents=True, exist_ok=True)
save_config(config, Path(config.checkpoint.drive_root) / 'experiment_config.json')
print('Config saved to Drive.')
print(f'Model params estimate: d_model={config.model.d_model}, slots={config.model.num_slots}, seq_len={config.model.max_seq_len}')
print(f'Training: {config.train.total_steps} steps, batch={config.train.global_batch_size}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Config saved to Drive.
Model params estimate: d_model=256, slots=6, seq_len=512
Training: 5000 steps, batch=32


In [9]:
# Cell 3: Prepare tokenizer and training shards
# Resumable: skips everything already done in Drive
# Dataset: flytech/python-codes-25k (fully open, no login required)

import sys, glob
from pathlib import Path

# Ensure repo is on path (safe to call even if already added)
REPO_ROOT = '/content/zap'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from datasets import load_dataset
import sentencepiece as spm
from cpmm_jax.data_pipeline import build_training_record, write_jsonl_shards

TOKENIZER_PREFIX = Path(config.data.tokenizer_path).with_suffix('')
TRAIN_DIR = Path(config.checkpoint.drive_root) / 'data' / 'train'
EVAL_DIR = Path(config.checkpoint.drive_root) / 'data' / 'eval'
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Dataset: open-source Python code, no login required
# Using multiple small fully-open datasets as fallback chain

def iter_python_code(max_files=15000):
    """Stream Python files from fully open datasets (no HF login needed)."""
    # Primary: flytech/python-codes-25k - fully open, real Python code
    sources = [
        ('flytech/python-codes-25k', None, 'train', 'output'),
        ('iamtarun/python_code_instructions_18k_alpaca', None, 'train', 'output'),
        ('TokenBender/code_instructions_122k_alpaca_style', None, 'train', 'output'),
    ]
    count = 0
    for dataset_name, config_name, split, code_field in sources:
        if count >= max_files:
            break
        try:
            ds = load_dataset(dataset_name, config_name, split=split, streaming=True)
            for row in ds:
                content = row.get(code_field, '') or row.get('code', '') or row.get('content', '') or row.get('instruction', '')
                if not content or len(content) < 64:
                    continue
                yield content, '', row.get('path', '')
                count += 1
                if count >= max_files:
                    break
            print(f'  loaded from {dataset_name}: total so far {count}')
        except Exception as e:
            print(f'  skipping {dataset_name}: {e}')
    # Fallback: generate synthetic Python code if datasets fail
    if count == 0:
        print('  using synthetic Python code fallback')
        templates = [
            'def add(a, b):\n    """Add two numbers."""\n    return a + b\n',
            'def factorial(n):\n    if n <= 1:\n        return 1\n    return n * factorial(n - 1)\n',
            'class Stack:\n    def __init__(self):\n        self.items = []\n    def push(self, item):\n        self.items.append(item)\n    def pop(self):\n        return self.items.pop()\n',
            'import os\nimport sys\n\ndef read_file(path):\n    with open(path) as f:\n        return f.read()\n',
            'def bubble_sort(arr):\n    n = len(arr)\n    for i in range(n):\n        for j in range(n - i - 1):\n            if arr[j] > arr[j + 1]:\n                arr[j], arr[j + 1] = arr[j + 1], arr[j]\n    return arr\n',
        ]
        for i in range(min(max_files, 5000)):
            yield templates[i % len(templates)], '', f'synthetic_{i}.py'
            count += 1

# --- Step 1: Train tokenizer (skip if already done) ---
if not Path(config.data.tokenizer_path).exists():
    print('Collecting Python code for tokenizer training...')
    text_dump = Path(config.checkpoint.drive_root) / 'tokenizer_corpus.txt'
    count = 0
    with text_dump.open('w', encoding='utf-8') as f:
        for code, _, _ in iter_python_code(max_files=8000):
            f.write(code.replace('\x00', ' ') + '\n')
            count += 1
            if count % 1000 == 0:
                print(f'  collected {count} files...')
    # Cap vocab size: SentencePiece requires vocab_size <= unique chars in corpus
    # 8000 Python files typically supports ~12000-15000 unique pieces
    actual_vocab_size = min(config.model.vocab_size, 12000)
    print(f'Training SentencePiece tokenizer on {count} files (vocab_size={actual_vocab_size})...')
    spm.SentencePieceTrainer.train(
        input=str(text_dump),
        model_prefix=str(TOKENIZER_PREFIX),
        vocab_size=actual_vocab_size,
        model_type='bpe',
        character_coverage=0.9995,
        bos_id=config.data.bos_id,
        eos_id=config.data.eos_id,
        pad_id=config.data.pad_id,
        unk_id=config.data.unk_id,
    )
    # Update config to match actual vocab size used
    config.model.vocab_size = actual_vocab_size
    print(f'Tokenizer saved with vocab_size={actual_vocab_size}.')
else:
    print('Tokenizer already exists, skipping.')

sp = spm.SentencePieceProcessor(model_file=config.data.tokenizer_path)

class SPAdapter:
    def __init__(self, p): self.p = p
    def encode(self, text): return self.p.encode(text, out_type=int)

tokenizer = SPAdapter(sp)
# Sync config vocab size with actual tokenizer (important for model init)
config.model.vocab_size = sp.get_piece_size()
print(f'Tokenizer ready. Vocab size: {config.model.vocab_size}')

# --- Step 2: Build training shards (stream-write, no RAM accumulation) ---
existing_train = glob.glob(str(TRAIN_DIR / '*.jsonl'))
if not existing_train:
    print('Building training shards (streaming directly to Drive)...')
    import json as _json

    EXAMPLES_PER_SHARD = 128
    MAX_TRAIN = 5000
    MAX_EVAL = 128

    class ShardWriter:
        def __init__(self, directory, prefix):
            self.directory = Path(directory)
            self.prefix = prefix
            self.shard_idx = 0
            self.handle = None
            self.count = 0
            self._open_next()

        def _open_next(self):
            if self.handle:
                self.handle.close()
            p = self.directory / f'{self.prefix}_{self.shard_idx:05d}.jsonl'
            self.handle = p.open('w', encoding='utf-8')
            self.shard_idx += 1

        def write(self, record, examples_per_shard):
            self.handle.write(_json.dumps(record) + '\n')
            self.count += 1
            if self.count % examples_per_shard == 0:
                self._open_next()

        def close(self):
            if self.handle:
                self.handle.close()

    train_writer = ShardWriter(TRAIN_DIR, 'train')
    eval_writer = ShardWriter(EVAL_DIR, 'eval')

    try:
        for code, hexsha, fpath in iter_python_code(max_files=MAX_TRAIN + MAX_EVAL + 500):
            record = build_training_record(
                code, tokenizer=tokenizer, config=config.data,
                metadata={'hexsha': hexsha, 'path': fpath},
            )
            if eval_writer.count < MAX_EVAL:
                eval_writer.write(record, EXAMPLES_PER_SHARD)
            elif train_writer.count < MAX_TRAIN:
                train_writer.write(record, EXAMPLES_PER_SHARD)
                if train_writer.count % 500 == 0:
                    print(f'  train={train_writer.count} eval={eval_writer.count}')
            if train_writer.count >= MAX_TRAIN:
                break
    finally:
        train_writer.close()
        eval_writer.close()

    print(f'Done. train={train_writer.count} eval={eval_writer.count}')
else:
    print('Shards already exist in Drive, skipping.')

train_count = len(glob.glob(str(TRAIN_DIR / '*.jsonl')))
eval_count = len(glob.glob(str(EVAL_DIR / '*.jsonl')))
print(f'\nTrain shards: {train_count}  |  Eval shards: {eval_count}')
print('Data preparation complete.')

Tokenizer already exists, skipping.
Tokenizer ready. Vocab size: 12000


SyntaxError: no binding for nonlocal 'train_shard_idx' found (1560118413.py, line 128)

In [ ]:
# Cell 4: Initialize model or resume from Drive checkpoint

import sys
if '/content/zap' not in sys.path:
    sys.path.insert(0, '/content/zap')

import jax
import numpy as np
from cpmm_jax.checkpointing import (
    create_checkpoint_manager, latest_step, load_lightweight_metadata,
    metadata_payload, restore_checkpoint, save_checkpoint, save_lightweight_metadata,
)
from cpmm_jax.data_pipeline import JsonlShardLoader, LoaderState, collate_batch
from cpmm_jax.train_step import create_train_state

rng = jax.random.PRNGKey(config.train.seed)
state, model = create_train_state(rng, config.model, config.cpmm, config.train)
total_params = sum(x.size for x in jax.tree_util.tree_leaves(state.params))
print(f'Model initialized: {total_params:,} parameters')

manager = create_checkpoint_manager(config.checkpoint, config.train.max_checkpoints_to_keep)
loader_state = LoaderState(rng_seed=config.train.seed)
global_step = 0
start_epoch = 0

# Auto-resume from latest Drive checkpoint
latest = latest_step(manager)
meta = load_lightweight_metadata(config.checkpoint)
if latest is not None and meta is not None:
    state, restored_meta = restore_checkpoint(manager, latest, state)
    global_step = int(restored_meta['step'])
    start_epoch = int(restored_meta['epoch'])
    loader_state = LoaderState(**restored_meta['loader_state'])
    print(f'Resumed from step {global_step} (epoch {start_epoch})')
else:
    print('Starting fresh training run from step 0')

train_loader = JsonlShardLoader(config.data.train_shards_glob, seed=config.train.seed)
eval_loader = JsonlShardLoader(config.data.eval_shards_glob, seed=config.train.seed + 1)
print(f'Train shards loaded: {len(train_loader.shards)}')
print(f'Eval shards loaded:  {len(eval_loader.shards)}')

In [ ]:
# Cell 5: CPMM pretraining loop (resumable)

import sys
if '/content/zap' not in sys.path:
    sys.path.insert(0, '/content/zap')

import time
import jax.numpy as jnp
from cpmm_jax.train_step import eval_step, train_step


def batch_iterator(loader, state_obj, batch_size):
    pending, current_state = [], state_obj
    for next_state, record in loader.iter_records(current_state):
        pending.append(record)
        current_state = next_state
        if len(pending) == batch_size:
            yield current_state, collate_batch(pending, config.model.max_seq_len, config.data.pad_id)
            pending = []


def to_device(batch):
    return {k: jnp.asarray(v) for k, v in batch.items()}


print(f'Starting training from step {global_step} to {config.train.total_steps}')
print('Checkpoints saved to Drive every', config.train.save_every, 'steps')
print()

train_iter = batch_iterator(train_loader, loader_state, config.train.global_batch_size)
t0 = time.time()

for step in range(global_step, config.train.total_steps):
    loader_state, batch = next(train_iter)
    batch = to_device(batch)
    state, metrics = train_step(state, model, batch, config.cpmm)

    if step % config.train.log_every == 0:
        elapsed = time.time() - t0
        steps_done = step - global_step + 1
        steps_per_sec = steps_done / max(elapsed, 1)
        eta_min = (config.train.total_steps - step) / max(steps_per_sec, 1e-6) / 60
        loss_str = f"loss={float(metrics['loss']):.4f} lm={float(metrics['lm_loss']):.4f}"
        print(f'step={step:5d} | {loss_str} | {steps_per_sec:.2f} steps/s | ETA {eta_min:.1f}min')

    if step % config.train.eval_every == 0 and step > global_step:
        _, eval_batch = next(batch_iterator(eval_loader,
                             LoaderState(rng_seed=config.train.seed + 1),
                             config.train.global_batch_size))
        eval_metrics = eval_step(state, model, to_device(eval_batch), config.cpmm)
        print(f'  [EVAL step={step}] loss={float(eval_metrics["loss"]):.4f} '
              f'lm={float(eval_metrics["lm_loss"]):.4f} '
              f'graph={float(eval_metrics["graph_structure_loss"]):.4f}')

    if step % config.train.save_every == 0 and step > global_step:
        payload = metadata_payload(
            step=step, epoch=start_epoch, rng_seed=config.train.seed,
            loader_state=loader_state,
            tokenizer_path=config.data.tokenizer_path,
            stage='pretrain',
        )
        save_checkpoint(manager, step, state, payload)
        save_lightweight_metadata(config.checkpoint, payload)
        print(f'  [SAVED checkpoint at step {step}]')

total_time = (time.time() - t0) / 60
print(f'\nPretraining complete. Total time: {total_time:.1f} min')
print(f'Final checkpoint saved to: {config.checkpoint.checkpoint_dir}')

In [ ]:
# Optional chat/code supervised fine-tuning

from cpmm_jax.finetune_chat import ChatExample, build_chat_corpus

chat_examples = [
    ChatExample(
        system=config.chat.system_prompt,
        user='Write a Python function that returns the Fibonacci sequence up to n.',
        assistant='def fib_upto(n):\n    seq = []\n    a, b = 0, 1\n    while a <= n:\n        seq.append(a)\n        a, b = b, a + b\n    return seq',
    ),
    ChatExample(
        system=config.chat.system_prompt,
        user='Explain what this code does: def add(a, b): return a + b',
        assistant='It defines a function named add that takes two arguments and returns their sum.',
    ),
]
chat_corpus = build_chat_corpus(chat_examples)
print(chat_corpus[0][:300])
print('Add your full instruction dataset and a chat fine-tuning loop here using the same checkpoint manager.')

In [ ]:
# Long-context code evaluation hooks

from cpmm_jax.eval_code_tasks import CodeEvalSample, aggregate_scores

samples = [
    CodeEvalSample(
        prompt='Find the definition used by helper() in a long file.',
        expected='def helper(x):\n    return x + 1',
        task_type='definition_lookup',
    ),
    CodeEvalSample(
        prompt='Trace the imported symbol used by process().',
        expected='from utils import normalize',
        task_type='import_reasoning',
    ),
]
predictions = [sample.expected for sample in samples]
aggregate_scores(samples, predictions)

## Resume notes

If Colab disconnects:
1. Reconnect to TPU runtime.
2. Run setup and Drive mount cells again.
3. Run the resume cell.
4. Run the pretraining loop cell again.

The notebook restores:
- model parameters
- optimizer state
- tokenizer path
- global step
- shard index
- sample offset
- training stage metadata